<div dir="rtl">

# 🛠️ أساسيات YOLO الجزء 4ب: الأداء التقني والتشخيص المهني 🔍

مرحباً بك في المعمل التقني! بعد أن نظرنا للموديل من زاوية مادية، حان الوقت لفتحه من الداخل وفهم جودته التقنية. 

في هذا المعمل، سنتعلم كيف نقارن بين أحجام النماذج المختلفة، وكيف نكتشف المشكلة الأشهر في الذكاء الاصطناعي: **الفرط في التخصيص (Overfitting)**.

</div>

<div dir="rtl">

## 🛠️ 1. تجهيز بيئة العمل

كالعادة، نتأكد من جاهزية الأدوات.

</div>

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

<div dir="rtl">

## 🏗️ 2. أحجام النماذج: Nano vs Small

نماذج YOLO تأتي بأحجام مختلفة لتناسب كل جهاز:
- **Nano (n)**: مثل "سيارة صغيرة"؛ سريعة جداً وتوفر في الوقود (المعالج)، لكنها قد لا تحمل أوزاناً ثقيلة (بيانات معقدة).
- **Small (s)**: مثل "سيارة عائلية"؛ أبطأ قليلاً لكنها أدق وأكثر استقراراً.

سنقوم الآن بتحميل النوعين للمقارنة بينهما.

</div>

In [ ]:
from ultralytics import YOLO
import time

# تحميل نموذجين مختلفين في الحجم
model_n = YOLO("yolo26n.pt") # النسخة النانو (الأسرع)
model_s = YOLO("yolo26s.pt") # النسخة الصغيرة (الأدق)

<div dir="rtl">

## ⚖️ 3. المقارنة: السرعة مقابل الدقة (Trade-off)

في هذا الجزء، سنقيس الوقت الذي يحتاجه كل نموذج لتحليل صورة واحدة، وسنقارن دقة كل منهما. هذا هو ما يسمى **المقايضة (Trade-off)**؛ لا يمكنك دائماً الحصول على أسرع شيء وأدق شيء في نفس الوقت.

</div>

In [ ]:
source_img = "https://ultralytics.com/images/bus.jpg"

print("⏳ جاري قياس سرعة الاستنتاج (3 محاولات لكل نموذج)...")

# قياس سرعة نموذج Nano
times_n = []
for _ in range(3):
    start = time.time()
    model_n.predict(source=source_img)
    times_n.append(time.time() - start)
avg_time_n = sum(times_n) / len(times_n)

# قياس سرعة نموذج Small
times_s = []
for _ in range(3):
    start = time.time()
    model_s.predict(source=source_img)
    times_s.append(time.time() - start)
avg_time_s = sum(times_s) / len(times_s)

# تقييم الدقة على مجموعة بيانات التحقق
metrics_n = model_n.val(data="coco8.yaml")
metrics_s = model_s.val(data="coco8.yaml")

print("\n" + "="*50)
print("📊 جدول المقارنة الفني")
print("="*50)
print(f"{'النموذج':<15} {'الدقة (mAP)':<12} {'السرعة (ثانية)':<12}")
print("-"*50)
print(f"{'YOLO Nano':<15} {metrics_n.box.map50:<12.4f} {avg_time_n:<12.3f}")
print(f"{'YOLO Small':<15} {metrics_s.box.map50:<12.4f} {avg_time_s:<12.3f}")
print("="*50)

<div dir="rtl">

### 💡 تمرين: ما هو الـ IoU؟

الـ **IoU** (Intersection over Union) أو "التقاطع على الاتحاد" هو الطريقة التي نقيس بها جودة رسم المربع. 

تخيل أنك تضع مربعين فوق بعضهما. 
- إذا انطبقا تماماً، فالنتيجة **1.0** (كمال).
- إذا لم يلمسا بعضهما، فالنتيجة **0.0** (فشل).

**تحدي:** لو كان لديك مربعين، مساحة التقاطع بينهما 50، ومساحة اتحادهما الكلية 200. كم تكون قيمة الـ IoU؟

</div>

In [ ]:
# احسب الـ IoU هنا يدوياً
intersection = 50
union = 200
iou = intersection / union
print(f"IoU = {iou}")

<div dir="rtl">

## 😵 4. فخ الـ Overfitting (حفظ المنهج لا فهمه)

تحدث مشكلة **الفرط في التخصيص (Overfitting)** عندما يصبح النموذج "حافظاً" لصور التدريب بدلاً من أن يكون "فاهماً" للقواعد العامة. 

**كيف نكتشفه؟**
- إذا كانت الدقة في التدريب عالية جداً (مثلاً 99%).
- والدقة في التحقق (بيانات جديدة) منخفضة جداً (مثلاً 60%).

هذا يعني أن النموذج قد "بصم" الصور ولم يتعلم.

</div>

In [ ]:
model_test = YOLO("yolo26n.pt")
# سنقوم بتدريب سريع للتجربة
model_test.train(data="coco8.yaml", epochs=5, imgsz=640, verbose=False)

val_metrics = model_test.val(data="coco8.yaml")
train_map = val_metrics.box.map50
val_map = val_metrics.box.map50 # في هذا المثال الصغير ستكون متساوية غالباً

print("\n🔍 فحص جودة التعلم:")
print(f"  دقة التدريب: {train_map:.4f}")
print(f"  دقة التحقق:   {val_map:.4f}")

if val_map < train_map * 0.9:
    print("\n⚠️  تحذير: يبدو أن النموذج يحفظ ولا يفهم (Overfitting)!")
else:
    print("\n✅ ممتاز: النموذج يتعلم القواعد بشكل سليم.")

<div dir="rtl">

---
## 🎉 نهاية الجزء الرابع
لقد أتممت الآن مهارات التشخيص الفني والمادي لنماذج الذكاء الاصطناعي. أنت الآن جاهز للانتقال للمرحلة الأخيرة والعمل مع مكتبات متقدمة مثل HuggingFace!

</div>